# 8주차 실습 — SQL (2) 집계와 JOIN

- **데이터**: `aact_mini` — 7주차와 같은 임상시험 37,509건입니다. 오늘은 `studies` 에 `countries` · `browse_conditions` · `facilities` 를 연결해 봅니다.
- **오늘 배우는 것** (8주차 학습 목표)
  - ① 집계 함수로 건수 · 평균 · 중앙값 · 비율을 구할 수 있다
  - ② GROUP BY로 묶어 세고, HAVING으로 그룹을 거를 수 있다
  - ③ 두 표를 JOIN으로 연결하고, 행 수 변화를 확인할 수 있다
- **사용법**: 따라 하기(1~3절: 집계 함수 → GROUP BY → COALESCE · CASE WHEN → HAVING → JOIN, 2절에 절 순서 맞추기 `O1`) → 4. 정리 → 직접 해보기 8문항. 각 절의 ‘자주 하는 실수’는 오류 메시지를 직접 보는 것이 목적입니다. 각 문항 아래 `check(...)` 를 실행하면 정답 여부가 바로 표시됩니다.
- **점수는 없습니다.** 7주차 노트북과 같은 방식으로 진행합니다.
- **과제 1 (20점)** — 7 · 8주차 범위, 9주차 수업 후 1주일 이내 LMS 제출. 오늘까지 배운 SELECT · WHERE · 집계 · GROUP BY · JOIN 이 범위입니다.

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title 자동 확인 도우미 — 정답 SQL 은 들어 있지 않습니다 (수정하지 마세요) { display-mode: "form" }
ANSWERS = _load_answers("eNqdlltvGkcUx7/KiBebKrJ22V0ufUsTmlhK4pY4raKqqoBdGkuWHRlHfYgiYXed4ksVSMDganFIiwOxqEwMjqlKWqn9Jn3cOXyHnDPL2rsUV3KfmD1zZjz/37n5cWBODnzMHgeyC9/ib0BTM1ImFksZRkQLXGGB5ZE9lNHVaFSWo8mQSvaV5e+yaNdwmV5epOVXgYcPklmDNu23PchXA1/j8sHC0iqdh/oa3yoxyFtgWrS6G78VvzbP/nn6nH2amLstFl/ejCfiYnUjMXfvM/bJffFx8+oXs3duiOVc4no84dpvzd6enWewv8G383y7McOg0gbziEHDhJrJhuUqX6/yZybjvw2gmWO8WbC7dQYvOmD24afSjBC4ohsrho5vXF15ZJDh4erC8lJyES2Z5GLWeIK2z/2MtJSWihi6kpR1L6Nx+4iR7GUEdROOe7TLt1t214T9PNR2PN8/b/ADP7prc/fuzE9DpReEWo7ZnedQ66HqPFEc7m7y5s4V5vhcn707P3sHqbrOxPqVxRAOP+g7R6v4Y2F0+MGAwcbmCJ2fhNB9EYqQF0U6rEXUmJyWDS3iRTFun4Ri+GzTPq3TLjSKUN7D5/l0y1DPs2HRgkoBdTDHnf11ys7cKcr2SZsCTSzKLVdX5w8om7zbG8XZPQv7Bf66DXu58ytGNruD6dEp8dwOJQqUt0DciR8if3obl+ekeDlFDU3RIiE1pRlJL6dxu8tJ8uXMtgX7Rah08C0XlBdfr0PTIu32aZ/v5ETU2+/AGvA3PSRjUaZQ3hCrX1pCbuGNWzf341cT084VQTrp/ClMLMbfIydzdNXEqvlPBqqXQSwkpeVoNBKRdF+upHQ1kpSVmBpJyR4GIS8C+9Tih60LxE/JksQPNxg/6uObpyjxydLtoTiMb5HSH/ODW797egUvvWXOOfJ3r0BE8P3aFIPD9+e+lxCseQWrKT2mS0ZGj6oxr2AlFpYiclLSVSXmDbpf8cmvGCVRHdvWsFL9V8QxJLs9KgwwMerHHbs7mBZtIIgFwzAHKOrwqkPAUKHorRR3qgzUVRuM+ip5Q2nN5WL3O9DI0YmYJOHZEZL/RSPspaFo4aSc1pOSaig+GmP2iV2zNuDd/gUgigXqEhThpfTqNws6Eym8/wNsnbiinA2RBI63QHTYoi/qwLs9ll19pC8Y2RnHFY6r/OWAPEk/fy0GCn+350kggdd5EfXjSpu/bBM9u5cbdSxomtSfC1V+JArS2cQZWPXHwbmF8eP8sNoX/atR5N0TvPOpuBrL7tKVF/GiTykhRdHTKU1SfEN93D6xS2PfGCduvqD8OGvNYrA7GpyZK+Q188O9KonFmhyuDVxCdJ+nCjsl+88SlBF1ue70KPMsMChabLj8HCzCwxno2KfFQHfHGJsWM0NUAR3HUZcTg/+wBbvF4GX4RX2pGw6n1Ew4qUf1qC91x+wTU3dy0p4jxInsTO+PggRyekTy78oIYZDBzo/khXd4/9dx3Igvnhqb/8P1htPILzvgn3wAN0p2kA==")

## 1. 집계로 통계 내기 — COUNT · AVG · 중앙값 · 비율

여러 행을 하나의 숫자로 만듭니다.

### 집계 함수 — 다섯 개면 대부분의 통계가 됩니다

| 함수 | 무엇을 구하나 | 예 | NULL은 |
|---|---|---|---|
| `COUNT` | 개수 | `COUNT(*)` | `COUNT(*)`는 포함, `COUNT(열)`은 제외 |
| `SUM` | 합계 | `SUM(enrollment)` | 제외하고 더함 |
| `AVG` | 평균 | `AVG(enrollment)` | 제외하고 나눔 |
| `MIN` / `MAX` | 가장 작은 / 큰 값 | `MAX(start_date)` | 제외 |
| `MEDIAN` | 중앙값 | `MEDIAN(enrollment)` | 제외 |

**주의**: 집계 함수는 NULL을 빼고 계산함.

### COUNT — COUNT(*) · COUNT(열) · COUNT(DISTINCT 열)

COUNT는 세 가지 모양이 있고, 결과가 서로 다릅니다.

| 쓰는 법 | 무엇을 세나 | studies에서 | 예시 5행 | 주의 |
|---|---|---:|---:|---|
| `COUNT(*)` | 모든 행 | 37,509 | 5 | NULL도 포함해서 셈 |
| `COUNT(열)` | 그 열에 값이 있는 행 | 4,352 | 3 | NULL은 세지 않음 |
| `COUNT(DISTINCT 열)` | 서로 다른 값의 개수 | 9 | 2 | 중복을 뺀 종류 수 |

예시 5행: 시험 A `2023-05-01` COMPLETED · 시험 B `NULL` COMPLETED · 시험 C `2022-11-10` RECRUITING · 시험 D `NULL` RECRUITING · 시험 E `2024-02-03` COMPLETED (`results_first_posted_date` · `overall_status`)

In [ ]:
%%sql
-- COUNT 세 가지 모양 → 37,509 / 4,352 / 9
SELECT COUNT(*) AS 전체행,
       COUNT(results_first_posted_date) AS 결과있음,
       COUNT(DISTINCT overall_status) AS 상태종류
FROM studies;

### 평균과 중앙값 — 극단값 하나가 평균을 크게 바꿉니다

In [ ]:
%%sql
-- “중재 연구의 등록 인원 평균 · 중앙값 · 최대는?”
SELECT AVG(enrollment) AS 평균,
       MEDIAN(enrollment) AS 중앙값,
       MAX(enrollment) AS 최대
FROM studies
WHERE study_type = 'INTERVENTIONAL';

In [ ]:
%%sql
-- ‘1만 명 이하’ 행: AND enrollment <= 10000
SELECT AVG(enrollment) AS 평균,
       MEDIAN(enrollment) AS 중앙값,
       MAX(enrollment) AS 최대
FROM studies
WHERE study_type = 'INTERVENTIONAL'
  AND enrollment <= 10000;

| | 평균 | 중앙값 | 최대 |
|---|---:|---:|---:|
| 전체 | 1,279 | 60 | 12,317,546 |
| 1만 명 이하 | 196 | 60 | 10,000 |

평균은 소수로 나옵니다(1278.58…). 반올림해서 읽으면 1,279 입니다.

치우친 값(재원일수 · 진료비 · 검사 수치)은 평균과 중앙값을 함께 보고합니다.

### 비율 구하기 — 퍼센트는 100.0 을 곱해서 계산합니다

In [ ]:
%%sql
-- “완료된 시험 중 결과가 게시된 비율은?” → 18.0
SELECT ROUND(100.0
         * COUNT(results_first_posted_date)
         / COUNT(*), 1) AS 결과게시율
FROM studies
WHERE overall_status = 'COMPLETED';

100 이 아니라 **100.0** 을 곱하는 이유: 정수끼리 나누면 소수점이 잘려 0이 나옵니다.

> 참고: 이 실습의 DuckDB 는 정수끼리 나누어도 소수로 계산해 줍니다. PostgreSQL 같은 다른 DB 에서는 소수점이 잘리므로(예: 3524 / 19622 → 0) 100.0 을 곱하는 습관을 들입니다.

## 2. GROUP BY와 HAVING — 묶어서 세고, 묶은 결과를 거르기

### SELECT 문의 절 — 7강의 다섯 절에 GROUP BY · HAVING 두 개가 더해집니다

| 절 | 무엇을 | 예 |
|---|---|---|
| `SELECT` | 무엇을 볼지 (열 · 집계) | `phase, COUNT(*)` |
| `FROM` | 어디에서 (테이블) | `studies` |
| `WHERE` | 어떤 행만 (묶기 전 조건) | `enrollment >= 100` |
| `GROUP BY` **NEW** | 무엇끼리 묶을지 (그룹) | `phase` |
| `HAVING` **NEW** | 어떤 그룹만 (묶은 뒤 조건) | `COUNT(*) >= 10` |
| `ORDER BY` | 어떤 순서로 (정렬) | `COUNT(*) DESC` |
| `LIMIT` | 몇 줄만 (개수) | `10` |

### GROUP BY — 묶어서 세기

**결과 한 행 = 그룹 하나** 입니다.

In [ ]:
%%sql
-- “연구 유형별로 몇 건씩 있어?”
SELECT study_type, COUNT(*) AS 건수,
       MEDIAN(enrollment) AS 중앙값
FROM studies
GROUP BY study_type
ORDER BY 건수 DESC;

묶는 열(`study_type`)은 SELECT에도 적어야 결과를 읽을 수 있습니다.

### 두 열로 묶기 — 교차표 만들기

GROUP BY 에 열을 더 적으면 됩니다.

In [ ]:
%%sql
-- “최근 2년, 연도별 · 연구 유형별 건수는?”
SELECT YEAR(start_date) AS 연도,
       study_type AS 유형,
       COUNT(*) AS 건수
FROM studies
WHERE start_date >= '2023-01-01'
GROUP BY 연도, 유형
ORDER BY 연도, 유형;

`YEAR(날짜)`는 연도만 꺼냅니다. 묶는 열을 여러 개 적으면 결과 한 행은 ‘연도 + 유형’ 조합 하나가 됩니다. 별칭(연도, 유형)으로 묶는 것은 DuckDB에서만 됩니다.

### 자주 하는 실수 — GROUP BY에 없는 열

> `[오류 예시]` 셀은 일부러 틀리게 적었습니다. 실행해서 오류 메시지를 직접 보고, 바로 아래 고친 셀과 비교하세요. 오류 메시지는 붉은 글씨 중 **`Error:` 가 들어간 줄**(예: `Binder Error: …`)만 읽으면 됩니다.

**하고 싶은 일** — 임상 단계별로, 진행 상태마다 시험이 몇 건인지 세기

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → Binder Error
SELECT phase, overall_status,
       COUNT(*)
FROM studies
GROUP BY phase

In [ ]:
%%sql
-- 이렇게 고칩니다
SELECT phase, overall_status,
       COUNT(*)
FROM studies
GROUP BY phase, overall_status

**왜 오류일까** — `GROUP BY phase` 로 묶으면 한 묶음에 상태가 여러 개라 무엇을 보여 줄지 정할 수 없습니다.

SELECT에 적은 열은 GROUP BY에 넣거나, COUNT 같은 집계 함수로 감쌉니다.

### 빈 값도 드러내기 — COALESCE

NULL 은 이름을 붙여야 보입니다.

In [ ]:
%%sql
-- “임상 단계별 건수를 보여 줘. 안 적힌 것도 같이.”
SELECT COALESCE(phase, '미기재') AS 단계,
       COUNT(*) AS 건수
FROM studies
GROUP BY 단계
ORDER BY 건수 DESC;

| 결과 | 값 · 찾는 법 | 뜻 |
|---|---|---|
| `NA` (16,695) | 글자 `'NA'` · `WHERE phase = 'NA'` | 단계 해당 없음 (N/A). 예: 기기 · 행동 연구 |
| `미기재` (9,602) | NULL (빈 칸) · `WHERE phase IS NULL` | 값이 비어 있음 → ‘미기재’로 표시됨 |

`'NA'`는 글자라서 IS NULL로 잡히지 않고, COALESCE로도 바뀌지 않습니다. 둘을 합쳐 세려면 CASE WHEN을 씁니다.

`GROUP BY 단계`(별칭)는 DuckDB 편의 기능이고, 표준 SQL은 `GROUP BY COALESCE(phase, '미기재')` 로 씁니다.

### CASE WHEN — 조건에 따라 나누어 세기

엑셀의 IF 와 같은 일을 합니다.

In [ ]:
%%sql
-- “등록 인원을 소 · 중 · 대규모로 나누어 세어 줘”
SELECT CASE WHEN enrollment < 50
            THEN '소규모'
            WHEN enrollment < 500
            THEN '중규모'
            ELSE '대규모'
       END AS 규모,
       COUNT(*) AS 건수
FROM studies
WHERE enrollment IS NOT NULL
GROUP BY 규모;

위에서부터 차례로 맞는 조건을 찾고, 맞으면 거기서 멈춥니다. 어디에도 안 맞으면 `ELSE`.

예시: enrollment 30 → 소규모, 120 → 중규모, 800 → 대규모

### HAVING — 집계 결과에 조건

WHERE는 묶기 전, HAVING은 묶은 뒤

In [ ]:
%%sql
-- “시험이 2000건 이상인 국가만 보여 줘”
SELECT name AS 국가,
       COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE removed = false
GROUP BY name
HAVING COUNT(DISTINCT nct_id) >= 2000
ORDER BY 시험수 DESC;

`WHERE removed = false` 는 묶기 전에 제외된 국가 행을 빼는 조건입니다.

국가별로 묶은 결과에서 HAVING이 그룹을 거릅니다 (2,000건 미만인 국가는 제외).

### 적는 순서와 처리 순서

7강의 처리 순서에서 WHERE와 SELECT 사이에 GROUP BY · HAVING이 들어갑니다. 집계에 대한 조건은 WHERE가 아니라 **HAVING**에 씁니다.

| | 순서 |
|---|---|
| 적는 순서 (사람의 순서) | SELECT › FROM › WHERE › GROUP BY › HAVING › ORDER BY › LIMIT |
| 처리 순서 (컴퓨터의 순서) | FROM › WHERE › GROUP BY › HAVING › SELECT › ORDER BY › LIMIT |

**왜 WHERE에는 집계 함수를 못 쓰나** — WHERE는 GROUP BY보다 먼저 처리됨. 아직 묶기 전이라 그룹마다의 COUNT(*)가 없음.

`WHERE COUNT(*) >= 10` → 오류 / `HAVING COUNT(*) >= 10` → 가능

**WHERE와 HAVING은 둘 다 거른다** — WHERE는 묶기 전에 행을 거름, HAVING은 묶은 뒤 그룹을 거름. 한 쿼리에 둘 다 쓸 수 있음. 행을 먼저 거르고, 묶은 결과를 다시 거름.

처리 과정으로 보면 WHERE는 FROM과 GROUP BY 사이에서 행을 거르고, HAVING은 GROUP BY 다음에 그룹을 거릅니다.

### 자주 하는 실수 — WHERE에 집계 함수 (실행하면 오류 메시지가 나옵니다)

In [ ]:
%%sql
-- [오류 예시] 이렇게 쓰면 → Binder Error (WHERE는 묶기 전이라 COUNT를 모릅니다)
SELECT phase, COUNT(*)
FROM studies
WHERE COUNT(*) > 1000
GROUP BY phase

In [ ]:
%%sql
-- 이렇게 고칩니다
SELECT phase, COUNT(*)
FROM studies
GROUP BY phase
HAVING COUNT(*) > 1000

집계에 대한 조건은 HAVING에 씁니다.

### 절 순서 맞추기 — 직접 해 보기

**O1.** “2024년에 시작한 시험을 임상 단계별로 세고, 많은 순으로 위에서 5줄만 보고 싶다.”

아래 셀의 여섯 조각은 슬라이드처럼 순서가 뒤섞여 있습니다. 먼저 그대로 실행해 붉은 글씨 중 `Error:` 가 들어간 줄(`Parser Error: syntax error at or near "LIMIT"`)을 보고, **줄의 순서만 바꾸어** 다시 실행한 뒤 `check` 를 실행하세요.

적는 순서: `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`

In [ ]:
%%sql
-- O1
-- [순서 맞추기] 순서가 뒤섞여 있어 그대로 실행하면 문법 오류가 납니다. 줄의 순서만 바꾸세요.
LIMIT 5
GROUP BY phase
SELECT phase, COUNT(*) AS 건수
ORDER BY 건수 DESC
FROM studies
WHERE start_date >= '2024-01-01'

In [ ]:
check("O1", _)

순서를 바꿔 적으면 문법 오류가 납니다. 외우기보다 이 줄을 옆에 적어 두고 쓰세요.

## 3. JOIN — 두 표를 연결하기

### JOIN이 필요한 이유 — 한 표에 없는 정보를 다른 표에서 가져오기

`studies` 표에는 국가 정보가 없습니다. 국가는 `countries` 표에 따로 있습니다.

![aact_mini 표 구조 한눈에 보기](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_erd.png)

7주차에 본 표 구조 그림입니다. `studies` 의 `nct_id`(PK)와 `countries` 의 `nct_id`(FK)로 연결합니다.

**공통으로 가진 열** — 두 표 모두 `nct_id` 가 있음. 두 표를 잇는 조건은 **ON** 절에 적습니다.

In [ ]:
%%sql
-- studies 와 countries 를 nct_id 로 연결 (앞의 4행만)
SELECT studies.nct_id, brief_title,
       name AS 국가
FROM studies
JOIN countries
  ON studies.nct_id = countries.nct_id
LIMIT 4;

In [ ]:
%%sql
-- 여러 나라에서 진행한 시험 하나만 보기 → 나라 수만큼 3행
SELECT studies.nct_id, brief_title,
       name AS 국가
FROM studies
JOIN countries
  ON studies.nct_id = countries.nct_id
WHERE studies.nct_id = 'NCT01672749';

여러 나라에서 진행한 시험은 나라 수만큼 행이 생깁니다. 위 시험은 `studies` 에서는 1행이지만, 연결하면 3행이 됩니다. (행이 얼마나 늘어나는지는 아래 ‘JOIN 뒤에는 행 수를 확인’에서 셉니다.)

### JOIN — 두 표 연결하기

질환별 시험 수 구하기

In [ ]:
%%sql
-- “뇌졸중 · 고혈압 · 천식은 각각 시험이 몇 건이야?”
SELECT mesh_term AS 질환,
       COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term IN ('Stroke',
                'Hypertension', 'Asthma')
GROUP BY mesh_term
ORDER BY 시험수 DESC;

질환 이름은 `conditions`(연구자 입력)가 아니라 `browse_conditions` 를 씁니다. MeSH = 의학 주제어 표준 목록. 같은 질환을 한 이름으로 부르기 위한 것입니다 (14주차 표준 용어).

### JOIN의 두 가지 — 양쪽에 다 있는 것만 vs 왼쪽은 모두

예시: studies 에 시험 A, 시험 B / facilities 에 A–서울병원, A–부산병원 → 시험 B는 기관 정보 없음

| JOIN (= INNER JOIN) | LEFT JOIN |
|---|---|
| 양쪽에 모두 있는 행만 남김 | 왼쪽(studies)은 모두 남김. 오른쪽에 없으면 NULL로 채움 |
| 시험 B가 빠짐 → 실제: 34,810건 | 시험 B는 NULL로 남음 → 실제: 37,509건 |

In [ ]:
%%sql
-- JOIN → 34,810건
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies JOIN facilities
  ON studies.nct_id = facilities.nct_id;

In [ ]:
%%sql
-- LEFT JOIN → 37,509건
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies LEFT JOIN facilities
  ON studies.nct_id = facilities.nct_id;

“전체 시험 중 몇 %”를 구할 때 INNER JOIN을 쓰면 **분모가 조용히 줄어듭니다.** 12주차에서 가장 자주 나오는 오류입니다.

### JOIN 뒤에는 행 수를 확인

한 시험에 질환이 여러 개 붙어 있습니다.

In [ ]:
%%sql
-- “JOIN하면 행이 몇 개가 되지?”
SELECT COUNT(*) AS JOIN후_행수,
       COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id;

행이 약 10배로 늘었습니다. 이 상태에서 `COUNT(*)`를 세면 시험 수가 아니라 (시험 × 질환) 쌍의 수입니다.

### 자주 하는 실수 — JOIN 뒤 COUNT(*)를 시험 수로 착각

오류가 나지 않아 더 위험합니다.

In [ ]:
%%sql
-- [오류 예시] 결과 4,098 — 오류는 없지만 이것은 기관 수입니다
SELECT COUNT(*) AS 한국시험수
FROM studies
JOIN facilities
  ON studies.nct_id = facilities.nct_id
WHERE country = 'South Korea'

In [ ]:
%%sql
-- 이렇게 고칩니다 → 1,182 (실제 시험 수)
SELECT COUNT(DISTINCT studies.nct_id)
         AS 한국시험수
FROM studies
JOIN facilities
  ON studies.nct_id = facilities.nct_id
WHERE country = 'South Korea'

한 시험이 여러 기관에서 진행되기 때문입니다. `COUNT(DISTINCT 키)`를 씁니다.

예시: 시험 A는 한국 기관 3곳, 시험 B는 1곳에서 진행 → JOIN하면 행이 4개 → `COUNT(*)` = 4 (기관 수), `COUNT(DISTINCT …)` = 2 (시험 수)

### 반정규화가 하는 일 — 그림으로 보기

나눠 둔 두 표를 JOIN으로 붙여 한 표로 만들면, 시험 정보가 질환 수만큼 반복됩니다.

`시험_질환` (넓은 표) — 한 행 = 시험 × 질환

| nct_id | brief_title | phase | mesh_term |
|---|---|---|---|
| 시험 A | 당뇨 신약 연구 | PHASE3 | Type 2 Diabetes |
| 시험 A | 당뇨 신약 연구 | PHASE3 | Obesity |
| 시험 A | 당뇨 신약 연구 | PHASE3 | Hypertension |
| 시험 B | 천식 흡입제 연구 | PHASE2 | Asthma |

- 시험 A의 제목을 고치려면 3행을 모두 바꿔야 합니다 (**갱신 이상**)
- 시험은 2건인데 행은 4개 → 실제 데이터는 31,634건 · 309,124행
- **왜 이렇게 하나** — 분석할 때 JOIN을 매번 쓰지 않아도 되어 편합니다. CDW의 넓은 표가 이 모양입니다.
- **대가** — 행이 늘고(시험 수 ≠ 행 수), 같은 값이 반복되어 고칠 때 어긋날 수 있습니다. 세는 단위를 꼭 확인합니다.

### 반정규화 실습 — 넓은 표 직접 만들기

나눠 둔 표를 JOIN으로 합쳐 한 표로 저장합니다. 슬라이드 SQL 에 `OR REPLACE` 만 붙여, 다시 실행해도 ‘이미 있는 표’ 오류가 나지 않게 했습니다.

In [ ]:
%%sql
-- 시험_질환 만들기
CREATE OR REPLACE TEMP TABLE 시험_질환 AS
SELECT studies.nct_id, brief_title,
       phase, mesh_term
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id;

In [ ]:
%%sql
-- 만든 표 확인 → 309,124행 · 31,634건 (바로 위 JOIN 결과와 같습니다)
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT nct_id) AS 서로다른_시험
FROM 시험_질환;

적힌 순서와 달리 ① FROM으로 studies를 가져와 ② JOIN · ON으로 같은 nct_id의 질환 행을 붙이고, ③ SELECT로 열 4개를 고른 뒤 ④ 새 표로 저장합니다.

`nct_id`는 두 표에 모두 있어 `studies.nct_id`처럼 표 이름을 붙이고, 나머지 열은 한 표에만 있어 이름만 씁니다.

TEMP 표는 노트북을 닫으면 사라집니다. 런타임을 다시 시작했다면 위의 `CREATE OR REPLACE TEMP TABLE 시험_질환` 셀부터 다시 실행하세요.

### 정규화 실습 — 넓은 표를 다시 나누기

반복되는 열을 떼어 내, 표마다 한 행의 단위가 하나가 되게 합니다. (예시) 6주차 2NF: nct_id에만 딸린 열을 떼어 냄

In [ ]:
%%sql
-- ① 시험마다 한 행
CREATE OR REPLACE TEMP TABLE 시험 AS
SELECT DISTINCT nct_id, brief_title, phase
FROM 시험_질환;

In [ ]:
%%sql
-- ② 시험 × 질환 한 쌍에 한 행
CREATE OR REPLACE TEMP TABLE 시험질환 AS
SELECT DISTINCT nct_id, mesh_term
FROM 시험_질환;

In [ ]:
%%sql
-- ③ 확인 → 31,634
SELECT COUNT(*) FROM 시험;

**확인할 것**
- **시험 표의 행 수가 31,634인가?** 넓은 표의 `COUNT(DISTINCT nct_id)`와 같아야 합니다. 이제 제목은 한 번만 저장됩니다.
- **37,509건이 돌아왔나?** 아닙니다. 질환이 없던 5,875건은 넓은 표에서 이미 사라졌습니다 (**삭제 이상**). 원본을 남겨 두는 이유입니다.

## 4. 정리 — 확인 습관 · 용어

### 집계와 JOIN을 쓸 때의 확인 습관 — 이 네 가지만 지켜도 대부분의 오류를 막습니다

- ① **무엇을 세는지 말로 정한다** — “시험 건수”인가 “기관 수”인가 “질환 쌍의 수”인가
- ② **JOIN 전후의 행 수를 비교한다** — `COUNT(*)` 와 `COUNT(DISTINCT 키)` 를 같이 본다
- ③ **분모가 줄지 않았는지 본다** — INNER JOIN 때문에 빠진 행이 있는지 LEFT JOIN으로 확인
- ④ **평균은 중앙값과 함께 본다** — 극단값 하나가 결과를 바꿀 수 있다

### 8주차 용어 정리 — 이번 시간에 새로 나온 말

| 용어 | 예시 | 뜻 (한 줄) |
|---|---|---|
| 집계 함수 | `AVG(enrollment)` | 여러 행을 하나의 값으로 만드는 함수 |
| COUNT(*) | `COUNT(*)` | 모든 행의 수 (NULL 포함) |
| COUNT(DISTINCT 열) | `COUNT(DISTINCT phase)` | 서로 다른 값의 개수 |
| MEDIAN | `MEDIAN(enrollment)` | 중앙값. 가운데 값이라 극단값에 덜 흔들림 |
| GROUP BY | `GROUP BY phase` | 같은 값끼리 묶는 절. 결과 1행 = 그룹 1개 |
| CASE WHEN | `CASE WHEN … THEN … END` | 조건에 따라 다른 값을 내놓기. 엑셀의 IF와 같음 |
| COALESCE | `COALESCE(phase, '미기재')` | 빈 값(NULL)을 정해 둔 값으로 바꾸기 |
| HAVING | `HAVING COUNT(*) >= 10` | 묶은 뒤의 그룹에 조건을 거는 절 |
| JOIN | `JOIN b ON …` | 양쪽에 모두 있는 행만 연결 (= INNER JOIN) |
| LEFT JOIN | `LEFT JOIN f ON …` | 왼쪽 표는 모두 남기고 연결 |
| ON | `ON s.nct_id = c.nct_id` | 두 표를 어떤 열로 맞출지 적는 부분 |
| 별칭 | `studies AS s` | 표나 열에 짧은 이름 붙이기 (AS) |

**별칭으로 쓰면** — 위 용어 표의 `ON s.nct_id = c.nct_id` 처럼, 표 이름 뒤에 `AS s` 를 붙이면 `studies.nct_id` 대신 `s.nct_id` 로 짧게 씁니다. 결과는 같습니다.

In [ ]:
%%sql
-- 표 별칭: studies AS s · facilities AS f (위 INNER JOIN 결과 34,810 과 같음)
SELECT COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN facilities AS f
  ON s.nct_id = f.nct_id;

## 직접 해보기 — 8문항

각 문항의 `%%sql` 셀에서 `___` 를 채워 실행한 뒤, **바로 다음 셀**의 `check(...)` 를 실행하세요. 두 번 틀리면 힌트가 나옵니다.
결과 열의 이름(`AS ...`)은 적힌 그대로 둡니다.

**Q1. COUNT 세 가지** — 임상 단계(`phase`)에 **값이 있는**(NULL이 아닌) 시험 수(`단계있음`, 글자 'NA' 도 값입니다)와, `phase` 값의 **종류 수**(`단계종류`)를 세세요.

In [ ]:
%%sql
-- Q1
SELECT COUNT(*) AS 전체,
       COUNT(___) AS 단계있음,
       COUNT(___) AS 단계종류
FROM studies;

In [ ]:
check("Q1", _)

> 7주차에 `SELECT DISTINCT phase` 로 본 값은 빈 값까지 9가지였는데, 여기서는 8입니다. COUNT(DISTINCT 열)은 빈 값(NULL)을 세지 않기 때문입니다.

**Q2. 평균과 중앙값** — 3상(`PHASE3`) 시험의 등록 인원 평균(`평균`, 반올림)과 중앙값(`중앙값`)을 구하세요. 두 값이 얼마나 다른지 보세요.

In [ ]:
%%sql
-- Q2
SELECT ROUND(___(enrollment)) AS 평균,
       ___(enrollment) AS 중앙값
FROM studies
WHERE phase = 'PHASE3';

In [ ]:
check("Q2", _)

> 평균이 중앙값의 4배가 넘습니다. 아주 큰 시험 몇 건이 평균을 끌어올렸기 때문입니다.

**Q3. GROUP BY** — 완료(`COMPLETED`)된 시험을 **시작 연도별로** 세세요. 결과 한 행 = 연도 하나입니다. (2절 ‘두 열로 묶기’ 예제의 `YEAR(start_date)` 를 참고하세요)

In [ ]:
%%sql
-- Q3
SELECT ___ AS 시작연도, COUNT(*) AS 건수
FROM studies
WHERE overall_status = 'COMPLETED'
GROUP BY ___
ORDER BY 시작연도;

In [ ]:
check("Q3", _)

**Q4. CASE WHEN** — 등록 인원을 '100명 미만'과 '100명 이상' 두 구간으로 나누어 세세요. `___` 에 조건을 적습니다.

In [ ]:
%%sql
-- Q4
SELECT CASE WHEN ___ THEN '100명 미만'
            ELSE '100명 이상'
       END AS 규모,
       COUNT(*) AS 건수
FROM studies
WHERE enrollment IS NOT NULL
GROUP BY 규모;

In [ ]:
check("Q4", _)

**Q5. HAVING** — 시험이 **900건 이상**인 국가만 보세요. `___` 자리에 조건을 한 줄로 적습니다. WHERE와 HAVING 중 어느 것일까요?

In [ ]:
%%sql
-- Q5
SELECT name AS 국가,
       COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE removed = false
GROUP BY name
___
ORDER BY 시험수 DESC;

In [ ]:
check("Q5", _)

**Q6. JOIN** — 일본(`Japan`) 기관에서 진행된 **시험 수**를 세세요. 두 표를 잇는 조건(ON)과 세는 방법을 채웁니다. (3절 ‘자주 하는 실수 — JOIN 뒤 COUNT(*)’를 참고하세요)

In [ ]:
%%sql
-- Q6
SELECT COUNT(___) AS 일본시험수
FROM studies
JOIN facilities
  ON ___
WHERE country = 'Japan';

In [ ]:
check("Q6", _)

**Q7. 넓은 표 ① 갱신 이상** — 시험 `NCT02369328` 의 제목을 고치려면 `시험_질환` 에서 몇 행을 바꿔야 할까요? 이 시험이 `시험_질환` 에 몇 행 있는지(`행수`) `___` 에 세는 식을 적으세요. (`시험_질환` 이 없다는 오류가 나면 3절의 `CREATE OR REPLACE TEMP TABLE 시험_질환` 셀을 먼저 실행하세요)

In [ ]:
%%sql
-- Q7
SELECT ___ AS 행수
FROM 시험_질환
WHERE nct_id = 'NCT02369328';

In [ ]:
check("Q7", _)

> 결과의 행 수만큼 제목을 모두 바꿔야 합니다. 한 행이라도 빠뜨리면 같은 시험의 제목이 서로 달라집니다 → **갱신 이상**. 정규화한 `시험` 표에서는 이 시험이 1행뿐이라 한 번만 고치면 됩니다.

**Q8. 넓은 표 ② 삭제 이상** — `시험_질환` 에 들어 있는 **서로 다른 시험 수**(`시험수`)를 세세요. `studies` 의 37,509건과 비교합니다. (`시험_질환` 이 없다는 오류가 나면 3절의 `CREATE OR REPLACE TEMP TABLE 시험_질환` 셀을 먼저 실행하세요)

In [ ]:
%%sql
-- Q8
SELECT COUNT(___) AS 시험수
FROM 시험_질환;

In [ ]:
check("Q8", _)

> 37,509 − 31,634 = **5,875건**은 질환 정보(`browse_conditions`)가 없어 이 표에 들어오지 못했습니다. `시험_질환` 만 남기고 나눠 둔 표를 지우면 이 시험들이 사라집니다 → **삭제 이상**.

## 오늘 배운 것

- GROUP BY 결과의 한 행이 무엇인지 말로 확인한다
- WHERE는 묶기 전, HAVING은 묶은 뒤 조건이다
- 처리는 FROM → WHERE → GROUP BY → HAVING → SELECT 순서다
- CASE WHEN으로 구간을 나누고, COALESCE로 빈 값에 이름을 붙인다
- JOIN 뒤에는 행 수가 변한다. COUNT(DISTINCT 키)로 확인한다

다음 시간: 9강. 의료데이터 SQL 실습 — 분석 질문을 SQL로 바꾸기, 과제 1 제출

아래 셀을 실행하면 오늘 확인한 문항(O1, Q1~Q8)을 한눈에 볼 수 있습니다.

In [ ]:
진행확인()